# 02 · pandas 3: dados tabulares

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

A partir deste notebook trabalharemos com um conjunto de dados real: 132 mil avaliações de produtos feitas por clientes de uma loja brasileira. Ele nos acompanha até o fim do curso. Aqui você aprende a carregá-lo, interrogá-lo e prepará-lo com o pandas, na versão 3, que mudou comportamentos importantes em relação ao que a maioria dos tutoriais ensina.

## O que você será capaz de fazer ao final

1. Carregar uma tabela, inspecionar tipos, tamanhos e valores ausentes.
2. Selecionar linhas e colunas com `loc`, `iloc`, máscaras e `query`.
3. Explicar o *Copy-on-Write* e reescrever código antigo que deixou de funcionar.
4. Trabalhar com texto, datas e categorias.
5. Resumir dados com `groupby`, `agg`, `transform` e tabelas cruzadas.
6. Combinar tabelas com `merge`, conferindo o resultado.
7. Escrever transformações encadeadas, rápidas e legíveis.

## Como usar este notebook

| Símbolo | Significado |
|---|---|
| 📌 | Lembrete de algo que você já viu e vai precisar agora |
| 🔮 | **Preveja antes de rodar**: responda mentalmente (ou por escrito) antes de executar a próxima célula |
| ✏️ | Exercício. A dificuldade vai de ★ (fixação) a ★★★ (desafio) |
| ✅ | Célula de verificação: confere se a sua resposta faz sentido. Não altere |
| ⚠️ | Armadilha comum |
| ⏳ | Célula que demora mais de 20 segundos |

Nos exercícios, substitua `...` pelo seu código. Se a célula ✅ terminar sem erro, você pode seguir. Se você travar em um exercício, pule: nenhuma célula posterior depende dele.

In [ ]:
import numpy as np
import pandas as pd

print("pandas", pd.__version__, "· NumPy", np.__version__)

if int(pd.__version__.split(".")[0]) < 3:
    raise RuntimeError(
        "Este notebook foi escrito para o pandas 3. Execute em uma célula:\n"
        "    %pip install -q -U 'pandas>=3,<4'\n"
        "depois use Ambiente de execução → Reiniciar sessão e execute o notebook desde o início."
    )

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.precision", 3)

## 1. Carregar e conhecer os dados

O **B2W-Reviews01** reúne avaliações feitas no site Americanas.com entre janeiro e maio de 2018.

> Real, L., Oshiro, M., Mafra, A. *B2W-Reviews01: an open product reviews corpus.* STIL 2019. Licença CC BY-NC-SA 4.0.

Três argumentos de `read_csv` evitam problemas logo na entrada: `dtype` fixa o tipo de colunas que parecem números e são identificadores; `parse_dates` converte texto em data; `low_memory=False` faz o pandas examinar a coluna inteira antes de decidir o tipo.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")
if not ARQUIVO.exists():
    urlretrieve(URL, ARQUIVO)

df = pd.read_csv(ARQUIVO, low_memory=False, dtype={"product_id": str}, parse_dates=["submission_date"])
print(df.shape)
df.head(3)

Um `DataFrame` é uma tabela em que cada coluna é uma `Series`: um *array* com um tipo e um **índice**, que dá nome às linhas. As quatro perguntas a fazer para qualquer tabela nova:

In [ ]:
df.info()                     # 1. Quais colunas, quais tipos, quantos valores não ausentes?

In [ ]:
df.isna().mean().sort_values(ascending=False).head(6).round(3)      # 2. Onde faltam dados?

In [ ]:
df.describe().round(2)        # 3. Os números fazem sentido?

In [ ]:
df.nunique().sort_values()    # 4. Quantos valores distintos há em cada coluna?

Só essas quatro saídas já contam bastante:

- a marca do produto falta em 69% das linhas; os dados do cliente, em 3% a 5%;
- o ano de nascimento vai de 59 a 2018: há valores impossíveis;
- há 113 mil clientes para 132 mil avaliações: a maioria avaliou uma única vez;
- as colunas de texto aparecem com o tipo `str`, novidade do pandas 3 (seção 4).

| Coluna | Conteúdo |
|---|---|
| `submission_date` | data e hora do envio |
| `reviewer_id`, `product_id` | identificadores do cliente e do produto |
| `product_name`, `product_brand` | nome e marca do produto |
| `site_category_lv1`, `site_category_lv2` | categoria e subcategoria |
| `review_title`, `review_text` | título e texto da avaliação |
| `overall_rating` | nota de 1 a 5 |
| `recommend_to_a_friend` | "Yes" ou "No" |
| `reviewer_birth_year`, `reviewer_gender`, `reviewer_state` | dados do cliente |

## 2. Selecionar linhas e colunas

| Quero | Escrevo |
|---|---|
| uma coluna (vira `Series`) | `df["overall_rating"]` |
| várias colunas (continua `DataFrame`) | `df[["overall_rating", "reviewer_state"]]` |
| linhas por condição | `df[df["overall_rating"] == 5]` |
| linhas **e** colunas, por rótulo | `df.loc[condição, ["coluna_a", "coluna_b"]]` |
| linhas e colunas, por posição | `df.iloc[0:5, 0:3]` |

In [ ]:
nota_baixa = df["overall_rating"] <= 2                 # máscara: uma Series de True/False
de_pernambuco = df["reviewer_state"] == "PE"

print("notas baixas:", nota_baixa.sum(), f"({nota_baixa.mean():.1%})")
df.loc[nota_baixa & de_pernambuco, ["submission_date", "product_name", "overall_rating"]].head(3)

As máscaras funcionam como no NumPy: combine com `&`, `|` e `~`, com parênteses. Três métodos poupam digitação:

In [ ]:
sudeste = df["reviewer_state"].isin(["SP", "RJ", "MG", "ES"])
intermediarias = df["overall_rating"].between(2, 4)                 # inclui as pontas
fala_de_entrega = df["review_text"].str.contains("entrega", case=False, na=False)

print(f"Sudeste: {sudeste.mean():.1%} · notas de 2 a 4: {intermediarias.mean():.1%} · citam 'entrega': {fala_de_entrega.mean():.1%}")

`query` aceita a condição como texto, o que fica mais legível em filtros longos. Variáveis do Python entram com `@`.

In [ ]:
nota_maxima = 2
df.query("overall_rating <= @nota_maxima and reviewer_state == 'PE'").shape

### ✏️ Exercício 1 (★): seleção

Crie `reclamacoes_celular`: as avaliações da categoria `"Celulares e Smartphones"` com nota 1, apenas com as colunas `product_name`, `review_title` e `reviewer_state`. Depois, guarde em `proporcao` a fração das avaliações dessa categoria que têm nota 1.

In [ ]:
# ✏️ Exercício 1
# SEU CÓDIGO AQUI
reclamacoes_celular = ...
proporcao = ...

In [ ]:
# ✅ Verificação do exercício 1 (não altere)
assert list(reclamacoes_celular.columns) == ["product_name", "review_title", "reviewer_state"], "selecione exatamente as três colunas, nessa ordem"
assert 3000 < len(reclamacoes_celular) < 6000, "confira as duas condições do filtro"
assert 0.15 < proporcao < 0.30, "a proporção é dentro da categoria, e não sobre a tabela inteira"
print(f"Exercício 1 correto: {len(reclamacoes_celular)} avaliações, {proporcao:.1%} da categoria.")

## 3. *Copy-on-Write*: a mudança mais importante do pandas 3

Até a versão 2, selecionar parte de uma tabela devolvia às vezes uma **visão** dos dados originais e às vezes uma **cópia**, conforme detalhes internos difíceis de prever. Alterar o resultado podia ou não alterar a tabela original, e o pandas emitia o célebre `SettingWithCopyWarning` quando ficava em dúvida.

O pandas 3 trocou isso por uma regra única, chamada *Copy-on-Write*:

> **Todo objeto obtido a partir de outro se comporta como uma cópia.** Alterar o objeto derivado nunca altera o original.

Por trás, o pandas só copia os dados de fato no momento em que alguém tenta alterá-los, e por isso a regra não custa memória. Vejamos as três consequências práticas.

### Consequência 1: mexer em um pedaço não altera a tabela

In [ ]:
pequena = df[["overall_rating", "reviewer_state"]].head(4).copy()

pedaco = pequena[pequena["overall_rating"] >= 4]
pedaco["overall_rating"] = 0                       # altera só o pedaço

print("pedaço:\n", pedaco, "\n\ntabela original, intacta:\n", pequena)

### Consequência 2: a atribuição encadeada nunca funciona

O padrão abaixo aparece em inúmeros tutoriais. Ele seleciona um pedaço (que é uma cópia) e atribui a essa cópia, que é descartada em seguida. No pandas 3 isso **não altera nada** e gera um aviso.

In [ ]:
import warnings

def avisos_de(acao):
    """Executa `acao` e mostra, em uma linha, os avisos que ela gerou."""
    with warnings.catch_warnings(record=True) as capturados:
        warnings.simplefilter("always")
        acao()
    for aviso in capturados:
        print(f"⚠️ {aviso.category.__name__}: {str(aviso.message).splitlines()[0]}")
    if not capturados:
        print("nenhum aviso")

def encadeada():
    pequena[pequena["overall_rating"] >= 4]["overall_rating"] = 0      # ⚠️ padrão antigo

avisos_de(encadeada)
print(pequena["overall_rating"].tolist(), "← nada mudou")

A forma correta faz seleção e atribuição em **um único passo**, com `loc`:

In [ ]:
pequena.loc[pequena["overall_rating"] >= 4, "overall_rating"] = 0
print(pequena["overall_rating"].tolist(), "← agora sim")

### Consequência 3: `inplace=True` em uma coluna selecionada não altera a tabela

Outro padrão comum em código antigo é selecionar uma coluna e pedir para alterá-la "no lugar". A coluna selecionada é um objeto derivado; a tabela não muda.

In [ ]:
com_ausente = pd.DataFrame({"estado": ["PE", None, "SP"]})

avisos_de(lambda: com_ausente["estado"].fillna("??", inplace=True))      # ⚠️ padrão antigo
print(com_ausente["estado"].tolist(), "← o ausente continua lá")

com_ausente["estado"] = com_ausente["estado"].fillna("??")                # forma correta: atribuir o resultado
print(com_ausente["estado"].tolist())

| Padrão antigo (não funciona mais) | Forma correta |
|---|---|
| `df[condição]["col"] = valor` | `df.loc[condição, "col"] = valor` |
| `df["col"][condição] = valor` | `df.loc[condição, "col"] = valor` |
| `df["col"].fillna(x, inplace=True)` | `df["col"] = df["col"].fillna(x)` |
| `parte = df[cond]` e depois alterar `parte` esperando mudar `df` | alterar `df` com `loc`, ou assumir que `parte` é independente |
| `.copy()` "por garantia" depois de cada filtro | desnecessário: o filtro já se comporta como cópia |

> ⚠️ O perigo está em código que **roda sem erro** e não faz o que se espera. Avisos de `ChainedAssignmentError` nunca devem ser ignorados.

### ✏️ Exercício 2 (★★): atualizar código legado

A função `legado` foi escrita para o pandas 1. Escreva `atualizado(tabela)`, que faz as mesmas três coisas da forma correta e **devolve uma tabela nova**, sem alterar a recebida:

1. preencher os estados ausentes com `"??"`;
2. nas linhas com nota 5, trocar o valor de `recommend_to_a_friend` por `"Yes"`;
3. criar a coluna `nota_dobrada`, com o dobro da nota.

In [ ]:
def legado(tabela):                                    # não altere: é o código a ser substituído
    tabela["reviewer_state"].fillna("??", inplace=True)
    tabela[tabela["overall_rating"] == 5]["recommend_to_a_friend"] = "Yes"
    tabela["nota_dobrada"] = tabela["overall_rating"] * 2
    return tabela

amostra = df[["reviewer_state", "overall_rating", "recommend_to_a_friend"]].sample(2000, random_state=1)

In [ ]:
# ✏️ Exercício 2
def atualizado(tabela):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 2 (não altere)
_antes = amostra.copy()
with warnings.catch_warnings():
    warnings.simplefilter("error")                     # qualquer aviso do pandas vira erro
    _res = atualizado(amostra)
assert amostra.equals(_antes), "a tabela recebida foi alterada"
assert _res["reviewer_state"].isna().sum() == 0 and (_res["reviewer_state"] == "??").sum() == _antes["reviewer_state"].isna().sum()
assert (_res.loc[_res["overall_rating"] == 5, "recommend_to_a_friend"] == "Yes").all()
assert (_res["nota_dobrada"] == _res["overall_rating"] * 2).all()
print("Exercício 2 correto.")

**Pergunta escrita.** Execute `legado(amostra.copy())` e observe o resultado. Quais das três tarefas a função antiga ainda realiza no pandas 3? Por que justamente essa(s)?

✏️ **Sua resposta:**

_Escreva aqui._

## 4. Texto: o tipo `str`

Até a versão 2, colunas de texto tinham o tipo genérico `object`, que aceita qualquer coisa. No pandas 3 elas passam a ter um tipo próprio, `str`, que só aceita texto ou ausente.

In [ ]:
print("tipo da coluna de texto:", df["review_text"].dtype)
print("é igual a object?", df["review_text"].dtype == object, "← testes antigos falham aqui")
print("forma correta de testar:", pd.api.types.is_string_dtype(df["review_text"]))
print("colunas de texto:", df.select_dtypes(include="str").columns.tolist())

In [ ]:
try:
    df.loc[df.index[0], "reviewer_state"] = 26            # um número em coluna de texto
except TypeError as erro:
    print("TypeError:", erro)

O tipo novo é mais rápido, ocupa menos memória e recusa valores que não são texto, o que revela cedo os erros de limpeza. O valor ausente em uma coluna de texto é `NaN`, o mesmo das colunas numéricas.

Os métodos de texto ficam sob `.str` e operam sobre a coluna inteira:

In [ ]:
texto = df["review_text"]

exemplos = pd.DataFrame({
    "original": texto.head(3),
    "minúsculas": texto.head(3).str.lower(),
    "tamanho": texto.head(3).str.len(),
    "palavras": texto.head(3).str.split().str.len(),
    "tem 'não'": texto.head(3).str.contains(r"\bnão\b", case=False, regex=True),
})
exemplos

> ⚠️ Com valores ausentes, `str.contains` devolve ausente, e uma máscara com ausentes não pode filtrar linhas. Use `na=False` para tratar o ausente como "não contém".

### ✏️ Exercício 3 (★): medidas do texto

Crie três `Series`, alinhadas com `df`:

1. `tamanho_texto`: número de caracteres de `review_text` (ausente onde o texto for ausente);
2. `grita`: verdadeiro quando o **título** está todo em maiúsculas (método `str.isupper`), falso para título ausente;
3. `cita_troca`: verdadeiro quando o texto contém "troca" ou "devolução", sem diferenciar maiúsculas, falso para texto ausente.

In [ ]:
# ✏️ Exercício 3
# SEU CÓDIGO AQUI
tamanho_texto = ...
grita = ...
cita_troca = ...

In [ ]:
# ✅ Verificação do exercício 3 (não altere)
assert len(tamanho_texto) == len(df) and tamanho_texto.isna().sum() == df["review_text"].isna().sum()
assert grita.dtype == bool and 0.03 < grita.mean() < 0.25, "grita deve ser booleana, sem ausentes"
assert cita_troca.dtype == bool and 0.03 < cita_troca.mean() < 0.15
print(f"Exercício 3 correto: mediana de {tamanho_texto.median():.0f} caracteres · {grita.mean():.1%} com título em maiúsculas · {cita_troca.mean():.1%} citam troca ou devolução.")
print("Nota média de quem cita troca ou devolução:", df.loc[cita_troca, "overall_rating"].mean().round(2), "· dos demais:", df.loc[~cita_troca, "overall_rating"].mean().round(2))

## 5. Criar e transformar colunas

### `assign` e `pd.col`

`assign` devolve uma tabela nova com colunas acrescentadas, sem alterar a original, e por isso combina bem com encadeamento. O pandas 3 trouxe `pd.col`, uma forma curta de se referir a uma coluna dentro dessas expressões.

In [ ]:
enriquecida = df.assign(
    nao_recomenda=pd.col("recommend_to_a_friend") == "No",
    tamanho_texto=pd.col("review_text").str.len(),
    idade=2018 - pd.col("reviewer_birth_year"),
)
enriquecida[["recommend_to_a_friend", "nao_recomenda", "tamanho_texto", "reviewer_birth_year", "idade"]].head(3)

### Condições, faixas e categorias

In [ ]:
idade = 2018 - df["reviewer_birth_year"]
idade = idade.where(idade.between(14, 100))                  # where: mantém onde a condição vale; o resto vira ausente

faixa = pd.cut(idade, bins=[13, 24, 39, 59, 100], labels=["14-24", "25-39", "40-59", "60+"])
sentimento = df["overall_rating"].map({1: "negativa", 2: "negativa", 3: "neutra", 4: "positiva", 5: "positiva"})

print(faixa.value_counts(dropna=False).sort_index(), "\n")
print(sentimento.value_counts(normalize=True).round(3))

`pd.cut` devolve uma coluna do tipo **categoria**: internamente, cada valor é um número pequeno que aponta para uma lista de rótulos. Converter colunas de texto com poucos valores distintos para esse tipo economiza muita memória.

In [ ]:
como_texto = df["site_category_lv1"]
como_categoria = como_texto.astype("category")

print(f"como texto:     {como_texto.memory_usage(deep=True) / 1e6:.2f} MB")
print(f"como categoria: {como_categoria.memory_usage(deep=True) / 1e6:.2f} MB  ({como_categoria.cat.categories.size} categorias)")

### Datas

Colunas de data têm o acessório `.dt`, análogo ao `.str`.

In [ ]:
data = df["submission_date"]
print("tipo:", data.dtype, "· de", data.min().date(), "a", data.max().date())

pd.DataFrame({
    "data": data.head(3),
    "mês": data.head(3).dt.month,
    "dia da semana": data.head(3).dt.day_name(),
    "hora": data.head(3).dt.hour,
    "semana": data.head(3).dt.to_period("W"),
})

> 📌 No pandas 3 as datas são guardadas, por padrão, com resolução de **microssegundos** (`datetime64[us]`), e não mais de nanossegundos. Código que converte datas em inteiros e pressupõe nanossegundos produz valores mil vezes menores.

### Juntando tudo: a tabela preparada

Vamos reunir as transformações em uma única cadeia e guardar o resultado em `dados`, a tabela que usaremos daqui em diante (e nos próximos notebooks).

Dois detalhes da cadeia abaixo merecem atenção:

- dentro do `assign`, `lambda t: ...` recebe a tabela **naquele ponto da cadeia**, já sem as linhas descartadas pelo `dropna`;
- `idade` foi calculada antes, a partir da tabela completa. Ao ser atribuída, o pandas **alinha pelo índice**: cada valor vai para a linha de mesmo rótulo, e os que sobram são ignorados. Esse alinhamento automático é a principal diferença em relação ao NumPy.

In [ ]:
idade = (2018 - df["reviewer_birth_year"]).where(lambda s: s.between(14, 100))

dados = (
    df
    .dropna(subset=["recommend_to_a_friend"])                 # 18 linhas sem resposta
    .assign(
        nao_recomenda=lambda t: (t["recommend_to_a_friend"] == "No").astype(int),
        idade=idade,
        faixa_etaria=pd.cut(idade, bins=[13, 24, 39, 59, 100], labels=["14-24", "25-39", "40-59", "60+"]),
        mes=lambda t: t["submission_date"].dt.month,
        texto=lambda t: t["review_title"].fillna("") + ". " + t["review_text"].fillna(""),
    )
)
print(dados.shape, "· a tabela original continua com", df.shape[1], "colunas")
dados[["nao_recomenda", "idade", "faixa_etaria", "mes", "texto"]].head(3)

### ✏️ Exercício 4 (★★): enriquecer a tabela

Escreva `enriquecer(tabela)`, que recebe uma tabela como `dados` e devolve uma tabela **nova** com quatro colunas acrescentadas:

- `nota_baixa`: verdadeiro quando a nota é 1 ou 2;
- `fim_de_semana`: verdadeiro quando o envio ocorreu no sábado ou no domingo (`dt.dayofweek` vale 5 e 6 nesses dias);
- `periodo`: a hora do envio classificada em `"madrugada"` (0 a 5), `"manhã"` (6 a 11), `"tarde"` (12 a 17) e `"noite"` (18 a 23), com `pd.cut`;
- `palavras`: o número de palavras da coluna `texto` (inteiro).

In [ ]:
# ✏️ Exercício 4
def enriquecer(tabela):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 4 (não altere)
_colunas_antes = list(dados.columns)
_e = enriquecer(dados)
assert list(dados.columns) == _colunas_antes, "a tabela recebida ganhou colunas: devolva uma tabela nova"
assert _e["nota_baixa"].dtype == bool and 0.26 < _e["nota_baixa"].mean() < 0.28
assert _e["fim_de_semana"].dtype == bool and 0.15 < _e["fim_de_semana"].mean() < 0.35
assert list(_e["periodo"].cat.categories) == ["madrugada", "manhã", "tarde", "noite"] and _e["periodo"].isna().sum() == 0
assert pd.api.types.is_integer_dtype(_e["palavras"]) and _e["palavras"].min() >= 1
print("Exercício 4 correto.")
_e.groupby("periodo", observed=True)["nota_baixa"].agg(["mean", "size"]).round(3)

## 6. Valores ausentes

Três decisões possíveis para um ausente: **descartar** a linha, **preencher** com um valor ou **manter** e deixar que as funções o ignorem. Não há resposta universal; a decisão depende do motivo da ausência e do uso que se fará da coluna.

In [ ]:
print("ausentes em idade:", dados["idade"].isna().sum())
print("média ignorando ausentes (padrão):", dados["idade"].mean().round(2))
print("após preencher com a mediana:     ", dados["idade"].fillna(dados["idade"].median()).mean().round(2))
print("linhas após descartar ausentes de idade e estado:", len(dados.dropna(subset=["idade", "reviewer_state"])))

> ⚠️ Preencher com a média ou a mediana **antes** de separar treino e teste usa informação do teste. No notebook 05 a imputação passa a ser um passo do modelo, aprendido só com o treino. Em análise exploratória, como aqui, preencher a tabela inteira não traz esse problema.

Antes de preencher, vale perguntar se a ausência é informativa:

In [ ]:
dados.assign(sem_idade=dados["idade"].isna()).groupby("sem_idade")["nao_recomenda"].agg(["mean", "size"]).round(3)

## 7. Agrupar e resumir

`groupby` responde a perguntas do tipo "para cada X, qual é o Y?". São três passos: **dividir** a tabela em grupos, **aplicar** um cálculo a cada grupo e **juntar** os resultados.

In [ ]:
dados.groupby("site_category_lv1")["overall_rating"].mean().sort_values().head(5).round(2)

Para várias medidas de uma vez, use a **agregação nomeada**: `nome_da_coluna_nova=("coluna_de_origem", "função")`.

In [ ]:
por_categoria = (
    dados
    .groupby("site_category_lv1")
    .agg(
        avaliacoes=("overall_rating", "size"),
        nota_media=("overall_rating", "mean"),
        taxa_nao_recomenda=("nao_recomenda", "mean"),
        produtos=("product_id", "nunique"),
    )
    .sort_values("avaliacoes", ascending=False)
)
por_categoria.head(6).round(3)

> ⚠️ `size` conta linhas; `count` conta valores não ausentes. Em colunas com ausentes os dois diferem.

### Contagens e tabelas cruzadas

In [ ]:
print(dados["overall_rating"].value_counts(normalize=True).sort_index().round(3), "\n")

pd.crosstab(dados["faixa_etaria"], dados["overall_rating"], normalize="index").round(3)      # cada linha soma 1

In [ ]:
dados.pivot_table(index="faixa_etaria", columns="reviewer_gender", values="nao_recomenda", aggfunc="mean", observed=True).round(3)

### `transform`: o resultado volta com o tamanho da tabela

`agg` devolve uma linha por grupo. `transform` devolve um valor **para cada linha original**, o que permite comparar cada linha com o seu grupo.

In [ ]:
media_da_categoria = dados.groupby("site_category_lv1")["overall_rating"].transform("mean")

comparacao = dados.assign(
    media_da_categoria=media_da_categoria,
    diferenca=dados["overall_rating"] - media_da_categoria,
)
comparacao[["site_category_lv1", "overall_rating", "media_da_categoria", "diferenca"]].head(4).round(2)

> ⚠️ Lembre-se do notebook 05 ao usar `transform` com o **alvo** de um modelo: uma coluna calculada com a média do alvo no grupo, incluindo a própria linha, é uma forma clássica de vazamento de dados.

### ✏️ Exercício 5 (★★): as piores subcategorias

Construa `piores`: uma tabela por `site_category_lv2` com as colunas `avaliacoes`, `nota_media` e `taxa_nao_recomenda`, contendo apenas as subcategorias com **pelo menos 1000 avaliações**, ordenada da maior para a menor `taxa_nao_recomenda`, e somente as **5 primeiras**.

In [ ]:
# ✏️ Exercício 5
# SEU CÓDIGO AQUI
piores = ...

In [ ]:
# ✅ Verificação do exercício 5 (não altere)
assert list(piores.columns) == ["avaliacoes", "nota_media", "taxa_nao_recomenda"] and len(piores) == 5
assert (piores["avaliacoes"] >= 1000).all(), "filtre as subcategorias com pelo menos 1000 avaliações"
assert piores["taxa_nao_recomenda"].is_monotonic_decreasing and piores["taxa_nao_recomenda"].iloc[0] > 0.35
print("Exercício 5 correto.")
piores.round(3)

### ✏️ Exercício 6 (★★): cada avaliação diante do seu produto

Crie a `Series` `avaliacoes_do_produto`, com o número de avaliações que o **produto daquela linha** recebeu, e a `Series` `acima_do_produto`, verdadeira quando a nota da linha é maior que a nota média do produto.

In [ ]:
# ✏️ Exercício 6
# SEU CÓDIGO AQUI
avaliacoes_do_produto = ...
acima_do_produto = ...

In [ ]:
# ✅ Verificação do exercício 6 (não altere)
assert len(avaliacoes_do_produto) == len(dados) == len(acima_do_produto), "use transform: um valor por linha"
assert avaliacoes_do_produto.min() == 1 and avaliacoes_do_produto.max() > 500
assert acima_do_produto.dtype == bool and not acima_do_produto[avaliacoes_do_produto == 1].any(), "produto com uma avaliação nunca fica acima da própria média"
print(f"Exercício 6 correto: {(avaliacoes_do_produto == 1).mean():.1%} das avaliações são de produtos avaliados uma única vez.")

## 8. Combinar tabelas

`merge` junta duas tabelas por uma ou mais colunas em comum, como o JOIN do SQL. Vamos acrescentar a região do país a cada avaliação.

In [ ]:
regioes = pd.DataFrame({
    "uf": ["AC", "AP", "AM", "PA", "RO", "RR", "TO", "AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE",
           "DF", "GO", "MT", "MS", "ES", "MG", "RJ", "SP", "PR", "RS", "SC"],
    "regiao": ["Norte"] * 7 + ["Nordeste"] * 9 + ["Centro-Oeste"] * 4 + ["Sudeste"] * 4 + ["Sul"] * 3,
})

com_regiao = dados.merge(
    regioes,
    left_on="reviewer_state", right_on="uf",
    how="left",               # mantém todas as avaliações, mesmo sem correspondência
    validate="m:1",           # exige que cada UF apareça uma única vez na tabela de regiões
    indicator=True,           # cria a coluna _merge, dizendo de onde veio cada linha
)
print("linhas antes:", len(dados), "· depois:", len(com_regiao))
com_regiao["_merge"].value_counts()

Três hábitos evitam os erros clássicos de junção:

1. **Compare o número de linhas antes e depois.** Se aumentou, havia chaves repetidas na tabela da direita, e as suas contagens estão infladas.
2. **Use `validate`** (`"1:1"`, `"m:1"`, `"1:m"`) para que o pandas confira a cardinalidade que você espera.
3. **Use `indicator=True`** e olhe as linhas sem correspondência (`left_only`). Aqui são as avaliações sem estado informado.

| `how` | Mantém |
|---|---|
| `"inner"` (padrão) | só as chaves presentes nas duas tabelas |
| `"left"` | todas as linhas da tabela da esquerda |
| `"outer"` | todas as linhas das duas |

### ✏️ Exercício 7 (★★): resumo por região

A partir de `com_regiao`, construa `por_regiao`: uma tabela indexada por `regiao`, com as colunas `avaliacoes`, `nota_media` e `taxa_nao_recomenda`, ordenada da maior para a menor taxa. As avaliações sem região ficam de fora.

In [ ]:
# ✏️ Exercício 7
# SEU CÓDIGO AQUI
por_regiao = ...

In [ ]:
# ✅ Verificação do exercício 7 (não altere)
assert set(por_regiao.index) == {"Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"}
assert por_regiao["avaliacoes"].sum() == com_regiao["regiao"].notna().sum()
assert por_regiao["taxa_nao_recomenda"].is_monotonic_decreasing
print("Exercício 7 correto.")
por_regiao.round(3)

**Pergunta escrita.** As taxas de não recomendação vão de cerca de 26% (Sul) a 28% (Norte). (a) Olhando o número de avaliações de cada região, essa diferença de dois pontos é grande o bastante para tirar conclusões? (b) Proponha uma hipótese para ela e diga que análise, com estes dados, ajudaria a testá-la.

✏️ **Sua resposta:**

_Escreva aqui._

## 9. O tempo como eixo

Com uma coluna de data no índice, `resample` agrupa por período, como um `groupby` feito para o tempo.

In [ ]:
semanal = (
    dados
    .set_index("submission_date")
    .resample("W")
    .agg(avaliacoes=("overall_rating", "size"), nota_media=("overall_rating", "mean"), taxa_nao_recomenda=("nao_recomenda", "mean"))
)

eixos = semanal[["avaliacoes", "taxa_nao_recomenda"]].plot(subplots=True, figsize=(8, 4.5), legend=False, title=["Avaliações por semana", "Taxa de não recomendação"])
eixos[1].set_xlabel("")
semanal.loc[semanal["taxa_nao_recomenda"].idxmax()].round(3)

O método `.plot` do pandas basta para olhar os dados durante a análise. O notebook 03 trata de gráficos com mais cuidado.

## 10. Encadeamento: uma análise em um parágrafo

Como quase todo método devolve uma tabela nova, é possível escrever uma análise como uma sequência de verbos, de cima para baixo, sem variáveis intermediárias. O método `pipe` encaixa funções suas na cadeia.

In [ ]:
def somente_com_volume(tabela, coluna, minimo):
    """Mantém as linhas cujo valor em `coluna` aparece pelo menos `minimo` vezes."""
    contagem = tabela.groupby(coluna)[coluna].transform("size")
    return tabela[contagem >= minimo]

resultado = (
    dados
    .query("overall_rating <= 2")                                   # 1. só as avaliações negativas
    .pipe(somente_com_volume, "site_category_lv1", 1000)            # 2. só as categorias com volume
    .assign(cita_entrega=lambda t: t["texto"].str.contains("entrega|prazo|atras", case=False, regex=True))
    .groupby("site_category_lv1")                                   # 3. para cada categoria...
    .agg(negativas=("overall_rating", "size"), citam_entrega=("cita_entrega", "mean"))
    .sort_values("citam_entrega", ascending=False)                  # 4. ...ordena pela proporção
    .head(5)
)
resultado.round(3)

Leia a cadeia em voz alta: "das avaliações negativas, em categorias com volume, qual proporção cita entrega, por categoria, da maior para a menor". Quando o código se lê como a pergunta, ele está bem escrito.

## 11. Desempenho

A regra é a mesma do NumPy: **opere sobre a coluna inteira**. Percorrer linhas, seja com `for`, com `iterrows` ou com `apply(axis=1)`, é dezenas ou centenas de vezes mais lento.

In [ ]:
import timeit

parte = dados.head(20_000)

def com_iterrows():
    return [linha["overall_rating"] * 2 if linha["nao_recomenda"] else linha["overall_rating"] for _, linha in parte.iterrows()]

def com_apply():
    return parte.apply(lambda linha: linha["overall_rating"] * 2 if linha["nao_recomenda"] else linha["overall_rating"], axis=1)

def vetorizado():
    return np.where(parte["nao_recomenda"] == 1, parte["overall_rating"] * 2, parte["overall_rating"])

tempos = {nome: min(timeit.repeat(f, number=1, repeat=2)) for nome, f in [("iterrows", com_iterrows), ("apply por linha", com_apply), ("vetorizado", vetorizado)]}
for nome, t in tempos.items():
    print(f"{nome:<16}{t * 1000:9.2f} ms   ({t / tempos['vetorizado']:6.0f}x)")

### ✏️ Exercício 8 (★★★): eliminar o `apply`

A função `rotulo_lento` classifica cada avaliação. Escreva `rotulo_rapido(tabela)`, que devolve uma `Series` com o mesmo resultado, sem `apply`, `iterrows` ou laços.

Dica: `np.select(lista_de_condições, lista_de_valores, default=...)` escolhe, para cada linha, o valor da primeira condição verdadeira.

In [ ]:
def rotulo_lento(tabela):
    def classificar(linha):
        if linha["overall_rating"] <= 2 and linha["nao_recomenda"] == 1:
            return "detrator"
        if linha["overall_rating"] >= 4 and linha["nao_recomenda"] == 0:
            return "promotor"
        if linha["overall_rating"] >= 4 and linha["nao_recomenda"] == 1:
            return "contraditório"
        return "neutro"
    return tabela.apply(classificar, axis=1)

In [ ]:
# ✏️ Exercício 8
def rotulo_rapido(tabela):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 8 (não altere)
_amostra = dados.sample(5000, random_state=0)
_lento, _rapido = rotulo_lento(_amostra), rotulo_rapido(_amostra)
assert isinstance(_rapido, pd.Series) and _rapido.index.equals(_amostra.index), "devolva uma Series com o mesmo índice da tabela"
assert (_rapido.to_numpy() == _lento.to_numpy()).all(), "o resultado difere do da função lenta"
_t_lento = min(timeit.repeat(lambda: rotulo_lento(_amostra), number=1, repeat=2))
_t_rapido = min(timeit.repeat(lambda: rotulo_rapido(_amostra), number=1, repeat=3))
assert _t_lento / _t_rapido > 5, "ainda há um laço escondido?"
print(f"Exercício 8 correto: {_t_lento / _t_rapido:.0f} vezes mais rápido.")
print(rotulo_rapido(dados).value_counts(normalize=True).round(3).to_string())

### Guardar em Parquet

CSV é texto: lento para ler e sem tipos (datas e categorias precisam ser reinterpretadas a cada leitura). **Parquet** é um formato binário por colunas, que guarda os tipos, comprime os dados e permite ler só as colunas necessárias. Nesta tabela, que é quase toda texto livre, o ganho de tamanho é modesto; o de tempo de leitura é grande.

In [ ]:
dados.to_parquet("avaliacoes.parquet")

t_csv = min(timeit.repeat(lambda: pd.read_csv(ARQUIVO, low_memory=False), number=1, repeat=2))
t_parquet = min(timeit.repeat(lambda: pd.read_parquet("avaliacoes.parquet"), number=1, repeat=2))
t_colunas = min(timeit.repeat(lambda: pd.read_parquet("avaliacoes.parquet", columns=["overall_rating", "reviewer_state"]), number=1, repeat=2))

print(f"CSV:     {ARQUIVO.stat().st_size / 1e6:5.1f} MB · leitura em {t_csv:.2f} s")
print(f"Parquet: {Path('avaliacoes.parquet').stat().st_size / 1e6:5.1f} MB · leitura em {t_parquet:.2f} s · só duas colunas em {t_colunas:.3f} s")
print("tipos preservados:", pd.read_parquet("avaliacoes.parquet")[["submission_date", "faixa_etaria"]].dtypes.astype(str).to_dict())

O arquivo `avaliacoes.parquet` contém a tabela preparada e será o ponto de partida dos próximos notebooks.

## 12. Para levar

### Erros comuns

| Erro | Como evitar |
|---|---|
| `df[cond]["col"] = valor` | `df.loc[cond, "col"] = valor` |
| `df["col"].metodo(inplace=True)` | `df["col"] = df["col"].metodo()` |
| Testar texto com `dtype == object` | `pd.api.types.is_string_dtype` ou `select_dtypes("str")` |
| `and` / `or` em máscaras | `&`, `|`, `~`, com parênteses |
| Máscara com ausentes vinda de `str.contains` | `na=False` |
| `merge` que multiplica linhas | conferir o tamanho; `validate=`; `indicator=True` |
| `apply(axis=1)` e `iterrows` | operações sobre colunas, `np.where`, `np.select` |
| Confundir `size` com `count` | `size` conta linhas; `count` ignora ausentes |
| Seguir tutorial ou código de assistente sem conferir a versão | informar a versão; não ignorar avisos |

### Resumo

- Comece toda tabela por `info`, ausentes, `describe` e `nunique`.
- `loc` seleciona e atribui em um passo; é a forma segura de alterar parte de uma tabela.
- No pandas 3, todo objeto derivado se comporta como cópia (*Copy-on-Write*) e texto tem o tipo `str`.
- `groupby` com agregação nomeada resume; `transform` devolve o resumo a cada linha.
- Toda junção merece conferência de tamanho e de cardinalidade.
- Encadear métodos produz código que se lê como a pergunta que ele responde.
- Opere sobre colunas inteiras e guarde resultados em Parquet.

### O que vem a seguir

No notebook 03 transformamos estas tabelas em gráficos: como escolher a forma certa para cada pergunta e como conduzir uma análise exploratória completa.

### Para casa

A lista de exercícios está em `02_casa.ipynb`.